# The boosting engine

The single-predictor engine drives `glmboost`/`gamboost`. It is a tight loop over
iterations; the per-step cost is dominated by one back-substitution per base
learner.

```{admonition} The math behind this loop
:class: tip
[](../methodology/worked_example) traces five iterations of a six-observation
fit by hand. This chapter traces its first iteration through the Rust
buffers and summarizes iterations 2–5 — see [](#the-worked-example-through-the-engine) below. Read the two
together: each abstract step (`offset`, `ngradient`, `fit_step`, `select`,
`update`) is applied to concrete buffers and numbers.
```

## The loop

Per iteration the engine:

1. Computes the negative gradient `u` (and the running risk) via the family's
   fused `ngradient_and_risk`, reusing the predictor buffer.
2. Calls `fit_step_ref(u, w)` on **every** prepared learner. Each returns its best
   coefficient, prediction, and weighted SSE against `u` — but the heavy matrix
   factorization was done once at preparation time, so this is a back-solve plus a
   matrix–vector product.
3. Selects the learner with the smallest SSE (ties broken by lowest index, to
   match R deterministically).
4. Updates the predictor in place: `f += nu * pred` of the winner, and records the
   winner's index, coefficient, and per-step increment.

The path stored is therefore: the scalar offset, and per iteration `(selected
index, coefficient vector, increment vector)`, plus the risk trajectory. Nothing
else is required to reconstruct the path. Work buffers are reused, while the
stored selections, coefficients, increments and risks grow with `mstop`.

## Factorizing once per learner

For a penalized learner the per-step solve is
$\hat\beta = (X^\top W X + \lambda K)^{-1} X^\top W u$. Across iterations only `u`
changes; $X$, $W$, $\lambda$, $K$ are fixed. So the engine forms and decomposes
the penalized Gram matrix $G = X^\top W X + \lambda K$ **once** (at `prepare_*`
time) and each step solves $G\hat\beta = X^\top W u$ by back-substitution — an
$O(p^2)$ operation instead of an $O(p^3)$ re-factorization. For `bols` (scalar
design) the "solve" collapses to a single division.

## In-place buffers

The engine pre-allocates the predictor `f`, the gradient `u`, and per-learner
scratch (`rhs_buf`, `beta_buf`, `pred_buf`) and reuses them every iteration.
`fit_step_ref` borrows these buffers and returns slices into them, so a 1000-step
fit performs no per-step heap allocation in the hot path. These buffers remove per-step heap allocation. Timing comparisons with the R
reference are reported in [](performance.ipynb).

```{figure} ../figures/mermaid-353bc6995d70.svg
:name: mermaid-353bc6995d70
:alt: Two-stage engine design: one-time allocation and Gram factorization followed by an allocation-free boosting loop that reuses scratch buffers.

Learner preparation allocates and factorizes once; each boosting iteration reuses the same buffers for gradient computation, learner fitting, and the selected update.
```

The diagram shows that preparation allocates buffers and factorizes the Gram
matrix; during the loop, functions borrow existing buffers or update them in
place. `fit_step_ref` returns slices
into the caller-owned buffers, so even the winner's $\beta$ and prediction do
not allocate — the engine holds references to them until the next iteration
overwrites them.

## Per-iteration cost

One general-loop iteration is $J$ per-learner solves plus one $O(n)$ `axpy`:

$$
T_{\text{iter}} \;=\; \underbrace{O(n)}_{\text{fused ngradient+risk}} \;+\;
\sum_{j=1}^{J} \underbrace{O(n p_j + p_j^2)}_{\text{matvec + back-solve}} \;+\;
\underbrace{O(n)}_{\text{axpy}}.
$$

The per-variant breakdown (prepare cost, per-step cost, predict cost) is in
[](base_learners#per-variant-cost-model). The headline numbers:

- `ScalarBols` (the `glmboost` case): each per-learner solve is $O(n)$ — a dot
  product and a division. With $J = 10^3$ candidates, one iteration is
  $\approx 10^3 n$ flops.
- `BlockBols` (the `gamboost` / `bbs` case): each per-learner solve is
  $O(n p_j + p_j^2)$ with $p_j \approx 20$ for a default spline — dominated by
  the matvec.
- Gaussian fast path: replaces the $\sum_j O(n p_j)$ term with
  $O(\sum_w p_w p_{j^\star})$ — a single selected learner — giving the
  factor-of-$J$ speedup described below.

## Selection and ties

Selection compares **weighted residual sums of squares**, exactly the quantity R
uses, including the tie-break rule (first learner wins). Deterministic tie-breaking makes the per-step selection path reproducible;
parity tests compare that path with R iteration by iteration.

## Risk recording

The risk is recorded after each full update so the path has length
$m_{\text{stop}}+1$ (index 0 is the offset-only risk). For families whose loss
couples observations (e.g. `CoxPH`), the risk is not a per-observation sum; those
families implement an $O(n)$ evaluation (a sorted cumulative pass over risk sets)
rather than the naive $O(n^2)$.

## Cross-validation reuse

`cvrisk` prepares learners under each fold's weights, because the
`df_to_lambda` solve depends on the weighted hat matrix and therefore changes when
out-of-bag observations receive weight zero ([](../methodology/penalties_df)).
The PyO3 `gamboost` CV preparer caches fold-invariant smooth-learner work
(`bbs`/`bspatial` bases, by-variable multiplication, tensor products, and
spectral centering) once per CV call, then recomputes only the fold-dependent
`λ`, weighted design transpose, and solver factorisation per fold. The fold
fits are independent and run in parallel.

For Gaussian CV, the Gram-update fast path is still available, but it is built
per fold. The full-sample Gram cannot be reused because fold weights change
both the Gram matrix and the `df_to_lambda` solution.

## The Gaussian fast path in detail

For Gaussian loss, the engine can bypass the $O(n)$ gradient computation
entirely by maintaining **per-learner right-hand sides** that are updated in
$O(p^2)$ via precomputed cross-Gram matrices. This is `try_fit_gaussian_gram`
in `engine.rs`.

### Preparation

Before the loop, precompute for each learner $j$:

- $G_j = B_j^\top W B_j + \lambda_j K_j$, the penalised Gram matrix, factorised
  via `GramSolver`.
- $R_j = B_j^\top W$, the $p_j \times n$ weighted design transpose, stored
  for computing initial RHS.
- $C_{jw} = B_j^\top W B_w$ for all learner pairs. The implementation computes
  only the upper triangle and fills the lower triangle by transposition, so the
  expensive matmul work is not duplicated. These are the **cross-Gram matrices**.

### The loop

**Initialize** RHS vectors $s_j = R_j (y - \hat f^{[0]})$ and the weighted
residual norm $\mathrm{wu2} = \sum_i w_i (y_i-\hat f^{[0]})^2$. The full
predictor vector is not maintained inside this loop.

At iteration $m$:

1. **Compute per-learner coefficients** by solving $G_j \beta_j = s_j$
   (back-substitution into the cached Cholesky). This gives the OLS fit of
   learner $j$ to the *current* pseudo-residual — without ever forming the
   residual vector explicitly.

2. **Compute per-learner SSE** as
   $$
   \text{SSE}_j =
   \mathrm{wu2} - \beta_j^\top s_j - \beta_j^\top P_j\beta_j,
   $$
   the residual-SSE identity for the penalised normal equations. This avoids
   forming the $n$-vector $B_j \beta_j$ for selection.

3. **Select** $j^\star = \argmin_j \text{SSE}_j$ (tie-break: lowest index).

4. **Update** the scalar residual norm:
   $$
   \mathrm{wu2} \gets \mathrm{wu2}
   - 2\nu \beta_{j^\star}^\top s_{j^\star}
   + \nu^2 \beta_{j^\star}^\top
     C_{j^\star j^\star}\beta_{j^\star}.
   $$

5. **Update all RHS vectors**:
   $$
   s_w \gets s_w - \nu C_{w j^\star} \beta_{j^\star}, \quad \forall w.
   $$
   The cross-Gram $C_{w j^\star} = B_w^\top W B_{j^\star}$ was precomputed, so
   this is a matrix-vector product of size $p_w \times p_{j^\star}$.

After the loop, the fitted vector is reconstructed once from the selected
learners and accumulated coefficients. That final reconstruction is $O(nP)$;
the iteration loop itself does not scan the full response vector.

### Conditions for the fast path

The fast path requires:

- **Gaussian family** — only the quadratic loss yields $u = y - f$, which is
  linear in $y$ and makes the RHS update exact.
- **Risk = "inbag"** — the in-bag risk is computed from the trace of the
  boosting hat matrix, not from a separate risk evaluation pass.
- **All learners are linear smoothers** — `btree` and `bkernel` have no fixed
  design matrix and cannot participate. If any such learner exists, the fast
  path is disabled and the general loop runs.
- **At least one block learner** — pure-scalar models (all `ScalarBols`) use
  the packed-scalar path instead, which is already near-optimal for $p_j = 1$.

The fast path activates unconditionally for any qualifying model — no
crossover guard on $\sum p_j$ vs $m_{\text{stop}}$. The $O(n \cdot P^2)$
cross-Gram setup cost (where $P = \sum_j p_j$) is amortised over
$m_{\text{stop}}$ iterations because each Gram iteration is $O(L \cdot
\bar{p}^2)$ — orders of magnitude cheaper than the general path's $O(L
\cdot \bar{p} \cdot n)$. Benchmark data on Apple M2 Pro ($n = 10^4$, $J
= 10$, $\bar p \approx 24$):

```{list-table}
:header-rows: 1

* - Path
  - $m_{\text{stop}}=200$
  - $m_{\text{stop}}=500$
* - Gram (cross-Gram cache)
  - 33 ms
  - 35 ms
* - General (per-iteration projection)
  - 80 ms
  - 204 ms
* - **Speedup**
  - **2.4×**
  - **5.8×**
```

For $J = 2$ the speedup is even larger ($\sim 18\times$) because the
cross-Gram setup is negligible ($2^2 = 4$ matrices). The setup cost grows
quadratically in $J$ but the per-iteration saving grows linearly, so the
break-even is reached for any realistic $m_{\text{stop}} \ge 50$.

The fast path reduces per-iteration cost from $O(nJ + \sum p_j^2)$ to
$O(\sum p_j^2 + \sum_w p_w p_{j^\star})$, where $j^\star$ is a single selected
learner. For spline-heavy models ($J \approx 10$, $p_j \approx 24$ for `bbs`),
the speedup is $2$–$6\times$ depending on $m_{\text{stop}}$, because the
$O(n \cdot P^2)$ cross-Gram setup must be amortised over enough iterations.



### Verifying the fast path: measured timings

The table above is a static snapshot. The claims it encodes — the Gram path
is faster overall, and its *marginal* per-iteration cost is far lower because
setup is amortised — are directly measurable: fit the same spline model twice
(once with `risk="inbag"`, which keeps the Gram fast path eligible, and once
with `risk="oobag"`, which forces the general path — see
[](performance.ipynb) for why this toggle is a clean way to compare the two
paths on identical data) at two different `mstop` values.

In [1]:
import time
import warnings
import numpy as np
import mboostrs as mb

warnings.filterwarnings("ignore")

rng = np.random.default_rng(0)
n, J = 10_000, 10
xs = {f"x{j}": rng.normal(size=n) for j in range(J)}
beta = rng.normal(scale=0.5, size=J)
y = sum(beta[j] * xs[f"x{j}"] for j in range(J)) + rng.normal(size=n)
data = {**xs, "y": y}
w = np.ones(n)
formula = "y ~ " + " + ".join(f"bbs(x{j}, df=4)" for j in range(J))


def time_fit(mstop, risk, reps=3):
    times = []
    for _ in range(reps):
        t0 = time.perf_counter()
        mb.gamboost(formula, data=data, family=mb.Gaussian(),
                    control=mb.boost_control(mstop=mstop, nu=0.1, risk=risk),
                    weights=w if risk == "oobag" else None)
        times.append(time.perf_counter() - t0)
    return min(times) * 1000  # ms, best-of-reps to cut scheduling noise


for mstop in (200, 500):
    t_gram = time_fit(mstop, "inbag")
    t_general = time_fit(mstop, "oobag")
    print(f"mstop={mstop:4d}   Gram: {t_gram:6.1f} ms   "
          f"General: {t_general:6.1f} ms   speedup: {t_general / t_gram:.1f}x")


mstop= 200   Gram:   24.5 ms   General:   60.3 ms   speedup: 2.5x


mstop= 500   Gram:   24.9 ms   General:  153.6 ms   speedup: 6.2x


In [2]:
# Isolate the *marginal* per-iteration cost from the one-time setup cost by
# comparing two mstop values far apart and dividing the time difference by
# the iteration difference. The readings are descriptive: the book build
# runs other notebooks at the same time, so a difference can come out small
# or even negative, and is then reported as unresolved rather than divided.
t_gram_lo, t_gram_hi = time_fit(200, "inbag"), time_fit(1000, "inbag")
t_gen_lo, t_gen_hi = time_fit(200, "oobag"), time_fit(1000, "oobag")

marginal_gram = (t_gram_hi - t_gram_lo) / (1000 - 200)
marginal_general = (t_gen_hi - t_gen_lo) / (1000 - 200)
print(f"Gram path:    {marginal_gram:.4f} ms/iteration marginal cost")
print(f"General path: {marginal_general:.4f} ms/iteration marginal cost")
if marginal_gram > 0 and marginal_general > 0:
    print(f"ratio: {marginal_general / marginal_gram:.0f}x")
else:
    print("ratio: not resolved by these timings (a difference was not positive)")


Gram path:    0.0010 ms/iteration marginal cost
General path: 0.2877 ms/iteration marginal cost
ratio: 275x


The total wall times above compare the two paths including their setup.
The marginal cost removes the setup and leaves the per-iteration work. In this
model, $n = 10{,}000$ rows and $J = 10$ `bbs` learners with $p_j = 24$ basis
columns each. The general path forms $B_j^\top W u$ for every learner on every
iteration, about $nJp_j = 2.4$ million multiply-adds. The Gram path solves each
learner's $24 \times 24$ system and updates the cross-products with the
selected learner's block, about $J p_j^2 + p_j \sum_k p_k \approx 12{,}000$
multiply-adds, independent of $n$. That factor of about 200 in operation counts
is the $O(nJ)$ against $O(\sum_j p_j^2)$ distinction derived above. The measured
ratio printed above is smaller and varies between runs, because both paths
also pay fixed per-iteration costs outside these products, and the four
timings share the machine with the rest of the book build. Repeatable timings
are in [](performance.ipynb).


### The general loop (non-Gaussian)

When the fast path is unavailable, each iteration:

1. **Compute the negative gradient** $u$ via the family's
   `ngradient_and_risk(y, f, w)` — an $O(n)$ fused pass that also returns the
   risk, sharing expensive link evaluations.
2. **For each learner**, compute $s_j = B_j^\top W u$ (a matrix-vector product,
   $O(p_j n)$), then solve $G_j \beta_j = s_j$ ($O(p_j^2)$). The coefficient
   and prediction are stored in preallocated buffers.
3. **Select** $j^\star$ by weighted SSE as before.
4. **Update** $f \mathrel{+}= \nu B_{j^\star} \beta_{j^\star}$.

Total per-iteration cost: $O(nJ + \sum p_j^2)$, dominated by $O(nJ)$ when
$J$ is large.

## The Rust loop, expressed as pseudocode

The general loop in `engine.rs` (`fit_with`) reads as follows:

```
fn fit_with(y, learners, family, control):
    n = len(y)
    f = vec![family.offset(y, w); n]         # allocate + fill with offset
    risk = vec![0.0; mstop + 1]               # risk[0] = offset risk
    risk[0] = family.risk(y, f, w)

    # pre-allocate per-iteration scratch
    u = vec![0.0; n]                          # negative gradient buffer
    rhs_buf = vec![0.0; max_p]               # right-hand side
    beta_buf = vec![0.0; max_p]              # coefficient
    pred_buf = vec![0.0; n]                  # increment prediction

    for m in 1..mstop:
        (u, risk[m]) = family.ngradient_and_risk(y, f, w)   # fused O(n)

        best_sse = INF; best_j = 0
        for (j, learner) in enumerate(learners):
            (beta, pred_j, sse) = learner.fit_step_ref(u, w,
                                      rhs_buf, beta_buf, pred_buf)
            if sse < best_sse - EPS:
                best_sse = sse; best_j = j
                # store refs to winner's data
                best_beta = beta; best_pred = pred_j

        # update predictor: f += nu * best_pred
        axpy(nu, best_pred, f)

        # record selection
        selected[m] = best_j
        coefs[best_j].push(best_beta)         # per-learner coefficient path

    return BoostingPath {
        offset: offset,
        selected: selected,
        coefficients: coefs,
        risk: risk,
        fitted: f,
        learner_store: learners
    }
```

`fit_step_ref` reuses caller-provided buffers (`rhs_buf`, `beta_buf`,
`pred_buf`) to avoid per-step allocation. The
buffer sizes are set once at preparation time based on the maximum
$p_j$ across all learners.

(the-worked-example-through-the-engine)=
## The worked example through the engine

This section traces the first iteration of the fit in
[](../methodology/worked_example) (six observations, `y ~ bols(x1) + bols(x2)`,
Gaussian, $\nu=0.1$) through the actual buffers and calls of `fit_with`. Read it
alongside the pseudocode above and the by-hand trace in the methodology chapter.

### Preparation (before the loop)

Each `bols(x)` becomes a `ScalarBols`. The preparer centers the column, caches
$W \tilde x$ and the scalar denominator $\tilde x^\top W \tilde x + \lambda$,
and bakes $\lambda$ (here $0$, because `df=1` on a centered column). After
preparation the engine holds:

```
learner[0] = ScalarBols { wx: [0.167,-0.833,0.167,1.167,-0.833,0.167] * w,
                          denom: 2.1389, mean_x: 0.8333 }     # bols(x1)
learner[1] = ScalarBols { wx: [...], denom: 2.1389, mean_x: 0.8333 }  # bols(x2)
```

The engine allocates the reusable buffers once, sized to the largest $p_j$
(here $p_j = 1$ for both learners):

```
f       = [1.75, 1.75, 1.75, 1.75, 1.75, 1.75]   # offset = mean(y)
u       = [0.0,  0.0,  0.0,  0.0,  0.0,  0.0 ]   # n=6 gradient buffer
rhs_buf, beta_buf, pred_buf                       # length-1 scratch
```

### Iteration 1

**Step 1 — fused gradient + risk.** `Gaussian::ngradient_and_risk(y, f, w)`
writes $u_i = y_i - f_i$ into `u` and returns the risk in one pass:

```
u  <- [0.25, -1.25, -0.25, 1.25, -0.75, 0.75]    # the pseudo-residual
risk[1] = 3.667
```

**Step 2 — `fit_step_ref` on each learner.** For `ScalarBols` the "solve"
collapses to one dot product and one division (no matrix, no back-substitution):

```
learner[0].fit_step_ref(u, w, rhs_buf, beta_buf, pred_buf):
    rhs  <- wx · u                 =  2.458      # into rhs_buf[0]
    beta <- rhs / denom            =  1.147      # into beta_buf[0]
    pred <- beta * wx              =  [0.192, -0.956, ...]   # into pred_buf
    sse  <- sum(w * (u - pred)^2)  =  0.647
    returns (beta=&beta_buf[0], pred=&pred_buf, sse=0.647)

learner[1].fit_step_ref(u, w, ...):   same arithmetic on x2
    beta <- -0.265 ; sse <- 4.177
```

Both learners wrote into the **same** scratch buffers, so only the
winner's values survive — but the engine has already captured each learner's
`sse` in a local scalar.

**Step 3 — select.** Compare the two SSEs and keep the winner's index:

```
best_sse = 0.647, best_j = 0   # bols(x1) wins; tie-break is lowest index
best_beta, best_pred borrowed from learner[0]'s fit_step_ref result
```

**Step 4 — update predictor in place:**

```
f[i] += nu * best_pred[i]      # axpy(0.1, best_pred, f)
f    = [1.769, 1.654, 1.769, 1.846, 1.654, 1.769]
selected[1] = 0 ; coefs[0].push(1.147)
```

This iteration uses one $O(n)$ fused gradient pass, two scalar learner solves
and one $O(n)$ `axpy`. The reusable work buffers and cached denominators avoid
allocating scratch arrays or factorizing a matrix during these operations.

### Iterations 2–5

The loop repeats verbatim. Because `x1` keeps winning, the `selected` array
fills with `0` and `coefs[0]` accumulates five entries
$(1.147, 1.032, 0.929, 0.836, 0.753)$. Their sum times $\nu$ is the final
aggregated coefficient $0.470$. `coefs[1]` stays empty — `x2` was never
selected, so its coefficient is exactly zero, with no explicit thresholding.

### Summary

- **Cached factorization.** `ScalarBols` has no matrix to factorize, but for `BlockBols` (`bbs`, `bspatial`, …) the `denom` above
  is a Cholesky factor computed once at `prepare_*`; each step's `beta <-
  rhs / denom` becomes an $O(p^2)$ back-substitution instead of an $O(p^3)$
  re-factorization. The loop structure is identical.
- **Selection criterion.** The engine selects learners by weighted SSE against
  the pseudo-residual; it does not evaluate `family.risk` for each candidate.
  The loop complexity is $O(\sum p_j^2 + nJ)$, with risk evaluated once per
  iteration rather than once per candidate.
- **Deterministic selection.** The lowest-index tie-break makes selection
  reproducible for identical computed SSEs. Parity tests compare the path
  with R `mboost` at the tolerances described in [](testing.ipynb);
  floating-point calculations can differ across implementations.

## See also

- [](architecture.ipynb) — where the engine fits in the crate layout and data flow.
- [](linear_algebra.ipynb) — the `GramSolver` and `df_to_lambda` kernels used by every step.
- [](../methodology/functional_gradient_boosting) — the conceptual algorithm the engine implements.
- [](../methodology/worked_example) — the by-hand trace that this chapter walks through the engine buffers.